# Ablation Study — FX Subset (35 classes)

Mục tiêu: Đánh giá định lượng từng đề xuất cải tiến được trình bày trong báo cáo, so sánh với Baseline.

## Sơ đồ thực nghiệm

| ID | Tên | Stream | Depth | Loss | Warm-up | Grad Clip | Weighted Sampler | Augment | Ghi chú |
|----|-----|--------|-------|------|---------|-----------|-----------------|---------|----------|
| E0 | Baseline | Single | 10 | CE | ✗ | ✗ | ✗ | ✗ | ST-GCN gốc |
| E1 | Focal Loss | Single | 6 | Focal | ✗ | ✗ | ✗ | ✗ | Giảm depth + Focal |
| E2 | + Warm-up & Clip | Single | 6 | Focal | ✓ | ✓ | ✗ | ✗ | Ổn định hóa training |
| E3 | + Weighted Sampler | Single | 6 | Focal | ✓ | ✓ | ✓ | ✗ | Cân bằng class |
| E4 | + Augment | Single | 6 | Focal | ✓ | ✓ | ✓ | ✓ | Tăng cường dữ liệu |
| E5 | Two-Stream | **Two** | 6 | Focal | ✓ | ✓ | ✓ | ✓ | **Full pipeline** |

**Lưu ý:**
- Tất cả cấu hình đều dùng `joint_spec=coco18`, `center_norm`, `bbox_norm`, apparatus=FX
- E0 là kết quả tham chiếu đã có từ Baseline-result.ipynb (không cần train lại)
- Cấu hình Two-Stream (E5) dùng `depth=6` như đã xác nhận qua depth search

In [ ]:
# ── Cell 1: Environment & paths ───────────────────────────────────────────────
import os
import subprocess
import sys
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'python-dotenv', '-q'], check=True)
    from dotenv import load_dotenv

_nb_dir = Path(globals().get('__vsc_ipynb_file__', __file__) if '__file__' in dir() else '.').resolve().parent
for _candidate in [_nb_dir / '.env', _nb_dir.parent / '.env', Path('.env')]:
    if _candidate.exists():
        load_dotenv(_candidate)
        print(f'Loaded .env from {_candidate}')
        break
else:
    print('No .env found — using Kaggle defaults')

REPO_DIR   = os.getenv('REPO_DIR',   '/kaggle/working/Yolo-ST-GCN')
BRANCH     = os.getenv('BRANCH',     'experiment-bonestream')
GYM288_PKL = os.getenv('GYM288_PKL', '/kaggle/working/Gym288-skeleton/gym_288_skeleton.pkl')
GYM99_PKL  = os.getenv('GYM99_PKL',  '/kaggle/working/Gym99-from-Gym288/gym99_from_gym288.pkl')
OUT_BASE   = os.getenv('OUT_DIR',    'outputs/ablation_fx')
POLICY_PATH = '/kaggle/working/fx_aug_policy.json'

# ── Common hyperparameters ─────────────────────────────────────────────────────
EPOCHS                  = 60
BATCH_SIZE              = 128
LR                      = '0.001'
WARMUP_EPOCHS           = '5'
EARLY_STOPPING_PATIENCE = '12'

EXP_IDS = ['E1', 'E2', 'E3', 'E4', 'E5']
EXP_DIRS = {eid: f'{OUT_BASE}/{eid}' for eid in EXP_IDS}

print(f'REPO_DIR  = {REPO_DIR}')
print(f'OUT_BASE  = {OUT_BASE}')
print(f'Experiments: {EXP_IDS}')

In [ ]:
# ── Cell 2: Repo setup ────────────────────────────────────────────────────────
REPO_URL = 'https://github.com/schizocatto/Yolo-ST-GCN.git'

if not Path(REPO_DIR).exists():
    print('Cloning repo...')
    subprocess.run(
        ['git', 'clone', '-b', BRANCH, '--single-branch', REPO_URL, REPO_DIR],
        check=True,
    )
else:
    print('Repo exists — pulling latest...')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Working dir:', os.getcwd())

In [ ]:
# ── Cell 3: Download Gym288 dataset ───────────────────────────────────────────
if Path(GYM288_PKL).exists():
    print(f'Gym288 pickle found: {GYM288_PKL}')
else:
    print('Downloading from HuggingFace...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'huggingface_hub', '-q'], check=True)
    from huggingface_hub import snapshot_download
    download_dir = Path(GYM288_PKL).parent
    download_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id='Lozumi/Gym288-skeleton',
        repo_type='dataset',
        local_dir=str(download_dir),
        local_dir_use_symlinks=False,
    )
    pkl_candidates = sorted(download_dir.rglob('*.pkl'))
    if not pkl_candidates:
        raise FileNotFoundError('No .pkl found after Gym288 download.')
    GYM288_PKL = str(pkl_candidates[0])
    print(f'Downloaded: {GYM288_PKL}')

In [ ]:
# ── Cell 4: Create FX augmentation policy ─────────────────────────────────────
import json

fx_policy = {
    "0": {
        "horizontal_flip_prob": 0.5,
        "scale_prob": 0.2,
        "scale_range": [0.95, 1.05],
        "random_shift": False,
        "random_move": False,
        "noise_std": 0.0,
        "joint_drop_prob": 0.0,
        "subsample_prob": 0.0
    },
    "1": {
        "horizontal_flip_prob": 0.5,
        "scale_prob": 0.5,
        "scale_range": [0.9, 1.1],
        "random_shift": True,
        "random_move": True,
        "move_angle": 5.0,
        "move_scale": 0.05,
        "move_trans": 0.0,
        "noise_std": 0.005,
        "joint_drop_prob": 0.05,
        "subsample_prob": 0.3,
        "subsample_factor_range": [0.9, 1.1]
    },
    "2": {
        "horizontal_flip_prob": 0.5,
        "scale_prob": 0.8,
        "scale_range": [0.85, 1.15],
        "random_shift": True,
        "random_move": True,
        "move_angle": 10.0,
        "move_scale": 0.1,
        "move_trans": 0.0,
        "noise_std": 0.01,
        "joint_drop_prob": 0.1,
        "subsample_prob": 0.5,
        "subsample_factor_range": [0.8, 1.2],
        "temporal_reverse_prob": 0.0
    }
}

Path(POLICY_PATH).parent.mkdir(parents=True, exist_ok=True)
with open(POLICY_PATH, 'w') as f:
    json.dump(fx_policy, f, indent=4)

print(f'Augmentation policy saved: {POLICY_PATH}')

---
## Phần 1: Huấn luyện các cấu hình

Các cell dưới đây lần lượt train từng cấu hình E1 → E5.
Mỗi cell **tự kiểm tra checkpoint** — nếu đã có thì bỏ qua, không cần chạy lại.

In [ ]:
# ── E1: Single-stream, depth=6, Focal Loss (không warm-up, không clip) ─────────
import importlib

OUT = EXP_DIRS['E1']
checkpoint = Path(OUT) / 'stgcn_gym99_coco18_d6_expert_FX.pth'
if checkpoint.exists():
    print(f'[E1] Checkpoint found, skipping: {checkpoint}')
else:
    sys.argv = [
        'train_gym99.py',
        '--auto_build_from_gym288',
        '--gym288_dataset_path',     GYM288_PKL,
        '--dataset_path',            GYM99_PKL,
        '--out_dir',                 OUT,
        '--apparatus',               'FX',
        '--epochs',                  str(EPOCHS),
        '--batch_size',              str(BATCH_SIZE),
        '--lr',                      LR,
        '--early_stopping_patience', EARLY_STOPPING_PATIENCE,
        '--num_workers',             '2',
        '--joint_spec_name',         'coco18',
        '--model_depth',             '6',
        '--loss_name',               'focal',
        '--focal_alpha_mode',        'sqrt_inverse',
        '--bbox_norm',
        '--center_norm',
        '--weight_decay',            '0.0005',
        '--save_every_epochs',       '10',
    ]
    import scripts.train_gym99 as _train
    importlib.reload(_train)
    print('>>> Training E1 — Single-stream depth=6, Focal Loss...')
    _train.main()
    print('\n✅ E1 done.')

In [ ]:
# ── E2: + Warm-up & Gradient Clipping ─────────────────────────────────────────
import importlib

OUT = EXP_DIRS['E2']
checkpoint = Path(OUT) / 'stgcn_gym99_coco18_d6_expert_FX.pth'
if checkpoint.exists():
    print(f'[E2] Checkpoint found, skipping: {checkpoint}')
else:
    sys.argv = [
        'train_gym99.py',
        '--auto_build_from_gym288',
        '--gym288_dataset_path',     GYM288_PKL,
        '--dataset_path',            GYM99_PKL,
        '--out_dir',                 OUT,
        '--apparatus',               'FX',
        '--epochs',                  str(EPOCHS),
        '--batch_size',              str(BATCH_SIZE),
        '--lr',                      LR,
        '--warmup_epochs',           WARMUP_EPOCHS,      # <-- Thêm warm-up
        '--early_stopping_patience', EARLY_STOPPING_PATIENCE,
        '--num_workers',             '2',
        '--joint_spec_name',         'coco18',
        '--model_depth',             '6',
        '--loss_name',               'focal',
        '--focal_alpha_mode',        'sqrt_inverse',
        '--bbox_norm',
        '--center_norm',
        '--grad_clip_norm',          '1.0',              # <-- Thêm gradient clipping
        '--weight_decay',            '0.0005',
        '--save_every_epochs',       '10',
    ]
    import scripts.train_gym99 as _train
    importlib.reload(_train)
    print('>>> Training E2 — + Warm-up + Gradient Clipping...')
    _train.main()
    print('\n✅ E2 done.')

In [ ]:
# ── E3: + Weighted Sampler ────────────────────────────────────────────────────
import importlib

OUT = EXP_DIRS['E3']
checkpoint = Path(OUT) / 'stgcn_gym99_coco18_d6_expert_FX.pth'
if checkpoint.exists():
    print(f'[E3] Checkpoint found, skipping: {checkpoint}')
else:
    sys.argv = [
        'train_gym99.py',
        '--auto_build_from_gym288',
        '--gym288_dataset_path',     GYM288_PKL,
        '--dataset_path',            GYM99_PKL,
        '--out_dir',                 OUT,
        '--apparatus',               'FX',
        '--epochs',                  str(EPOCHS),
        '--batch_size',              str(BATCH_SIZE),
        '--lr',                      LR,
        '--warmup_epochs',           WARMUP_EPOCHS,
        '--early_stopping_patience', EARLY_STOPPING_PATIENCE,
        '--num_workers',             '2',
        '--joint_spec_name',         'coco18',
        '--model_depth',             '6',
        '--loss_name',               'focal',
        '--focal_alpha_mode',        'sqrt_inverse',
        '--bbox_norm',
        '--center_norm',
        '--grad_clip_norm',          '1.0',
        '--use_weighted_sampler',                        # <-- Thêm weighted sampler
        '--weight_decay',            '0.0005',
        '--save_every_epochs',       '10',
    ]
    import scripts.train_gym99 as _train
    importlib.reload(_train)
    print('>>> Training E3 — + Weighted Sampler...')
    _train.main()
    print('\n✅ E3 done.')

In [ ]:
# ── E4: + Augmentation (SkeletonFeeder) ───────────────────────────────────────
import importlib

OUT = EXP_DIRS['E4']
checkpoint = Path(OUT) / 'stgcn_gym99_coco18_d6_expert_FX.pth'
if checkpoint.exists():
    print(f'[E4] Checkpoint found, skipping: {checkpoint}')
else:
    sys.argv = [
        'train_gym99.py',
        '--auto_build_from_gym288',
        '--gym288_dataset_path',     GYM288_PKL,
        '--dataset_path',            GYM99_PKL,
        '--out_dir',                 OUT,
        '--apparatus',               'FX',
        '--epochs',                  str(EPOCHS),
        '--batch_size',              str(BATCH_SIZE),
        '--lr',                      LR,
        '--warmup_epochs',           WARMUP_EPOCHS,
        '--early_stopping_patience', EARLY_STOPPING_PATIENCE,
        '--num_workers',             '2',
        '--joint_spec_name',         'coco18',
        '--model_depth',             '6',
        '--loss_name',               'focal',
        '--focal_alpha_mode',        'sqrt_inverse',
        '--bbox_norm',
        '--center_norm',
        '--grad_clip_norm',          '1.0',
        '--use_weighted_sampler',
        '--use_augment_feeder',                          # <-- Thêm SkeletonFeeder augment
        '--aug_config_path',         POLICY_PATH,
        '--weight_decay',            '0.0005',
        '--save_every_epochs',       '10',
    ]
    import scripts.train_gym99 as _train
    importlib.reload(_train)
    print('>>> Training E4 — + SkeletonFeeder Augmentation...')
    _train.main()
    print('\n✅ E4 done.')

In [ ]:
# ── E5: Two-Stream (Full pipeline) ────────────────────────────────────────────
import importlib

OUT = EXP_DIRS['E5']
checkpoint = Path(OUT) / 'stgcn_gym99_coco18_2s_d6_expert_FX.pth'
if checkpoint.exists():
    print(f'[E5] Checkpoint found, skipping: {checkpoint}')
else:
    sys.argv = [
        'train_gym99.py',
        '--auto_build_from_gym288',
        '--gym288_dataset_path',     GYM288_PKL,
        '--dataset_path',            GYM99_PKL,
        '--out_dir',                 OUT,
        '--apparatus',               'FX',
        '--epochs',                  str(EPOCHS),
        '--batch_size',              str(BATCH_SIZE),
        '--lr',                      LR,
        '--warmup_epochs',           WARMUP_EPOCHS,
        '--early_stopping_patience', EARLY_STOPPING_PATIENCE,
        '--num_workers',             '2',
        '--joint_spec_name',         'coco18',
        '--use_two_stream',                              # <-- Two-Stream
        '--model_depth',             '6',
        '--loss_name',               'focal',
        '--focal_alpha_mode',        'sqrt_inverse',
        '--bbox_norm',
        '--center_norm',
        '--grad_clip_norm',          '1.0',
        '--use_weighted_sampler',
        '--use_augment_feeder',
        '--aug_config_path',         POLICY_PATH,
        '--weight_decay',            '0.0005',
        '--save_every_epochs',       '10',
    ]
    import scripts.train_gym99 as _train
    importlib.reload(_train)
    print('>>> Training E5 — Two-Stream (Full pipeline), depth=6...')
    _train.main()
    print('\n✅ E5 done.')

---
## Phần 1b: Two-Stream bare — chưa có optimization techniques

Mục tiêu: Kiểm tra xem bản thân kiến trúc Two-Stream (không warm-up, không clip,
không weighted sampler, không augment) đã vượt trội hơn Baseline hay chưa.
Thử trên 3 độ sâu 8 / 6 / 4 để thấy ảnh hưởng của depth trong Two-Stream.

| ID  | Stream | Depth | Loss  | Warm-up | Grad Clip | Sampler | Aug |
|-----|--------|-------|-------|---------|-----------|---------|-----|
| E6a | Two    | 8     | Focal | ✗       | ✗         | ✗       | ✗   |
| E6b | Two    | 6     | Focal | ✗       | ✗         | ✗       | ✗   |
| E6c | Two    | 4     | Focal | ✗       | ✗         | ✗       | ✗   |

In [ ]:
# ── Config cho nhóm E6 (Two-Stream bare) ─────────────────────────────────────
E6_DEPTHS = [8, 6, 4]
E6_DIRS   = {d: f'{OUT_BASE}/E6_2s_d{d}_bare' for d in E6_DEPTHS}
E6_IDS    = {d: f'E6{chr(ord("a") + i)}' for i, d in enumerate(E6_DEPTHS)}  # E6a, E6b, E6c

# Cập nhật vào dict tổng EXP_DIRS và EXP_LABELS để dùng ở phần so sánh
for d in E6_DEPTHS:
    eid = E6_IDS[d]
    EXP_DIRS[eid]   = E6_DIRS[d]
    EXP_LABELS[eid] = f'{eid} — Two-Stream bare (depth={d})'
    COLORS[eid]     = {8: '#8e44ad', 6: '#9b59b6', 4: '#d7bde2'}[d]  # tím nhạt/đậm
    STYLES[eid]     = {8: '--', 6: '-.', 4: ':'}[d]
    WIDTHS[eid]     = 1.8

print('E6 dirs:')
for d in E6_DEPTHS:
    print(f'  {E6_IDS[d]}: {E6_DIRS[d]}')

In [ ]:
# ── E6a: Two-Stream bare, depth=8 ─────────────────────────────────────────────
import importlib

DEPTH = 8
EID   = E6_IDS[DEPTH]
OUT   = E6_DIRS[DEPTH]
checkpoint = Path(OUT) / f'stgcn_gym99_coco18_2s_d{DEPTH}_expert_FX.pth'
if checkpoint.exists():
    print(f'[{EID}] Checkpoint found, skipping: {checkpoint}')
else:
    sys.argv = [
        'train_gym99.py',
        '--auto_build_from_gym288',
        '--gym288_dataset_path',     GYM288_PKL,
        '--dataset_path',            GYM99_PKL,
        '--out_dir',                 OUT,
        '--apparatus',               'FX',
        '--epochs',                  str(EPOCHS),
        '--batch_size',              str(BATCH_SIZE),
        '--lr',                      LR,
        '--early_stopping_patience', EARLY_STOPPING_PATIENCE,
        '--num_workers',             '2',
        '--joint_spec_name',         'coco18',
        '--use_two_stream',
        '--model_depth',             str(DEPTH),
        '--loss_name',               'focal',
        '--focal_alpha_mode',        'sqrt_inverse',
        '--bbox_norm',
        '--center_norm',
        '--weight_decay',            '0.0005',
        '--save_every_epochs',       '10',
    ]
    import scripts.train_gym99 as _train
    importlib.reload(_train)
    print(f'>>> Training {EID} — Two-Stream bare depth={DEPTH}...')
    _train.main()
    print(f'\n✅ {EID} done.')

In [ ]:
# ── E6b: Two-Stream bare, depth=6 ─────────────────────────────────────────────
import importlib

DEPTH = 6
EID   = E6_IDS[DEPTH]
OUT   = E6_DIRS[DEPTH]
checkpoint = Path(OUT) / f'stgcn_gym99_coco18_2s_d{DEPTH}_expert_FX.pth'
if checkpoint.exists():
    print(f'[{EID}] Checkpoint found, skipping: {checkpoint}')
else:
    sys.argv = [
        'train_gym99.py',
        '--auto_build_from_gym288',
        '--gym288_dataset_path',     GYM288_PKL,
        '--dataset_path',            GYM99_PKL,
        '--out_dir',                 OUT,
        '--apparatus',               'FX',
        '--epochs',                  str(EPOCHS),
        '--batch_size',              str(BATCH_SIZE),
        '--lr',                      LR,
        '--early_stopping_patience', EARLY_STOPPING_PATIENCE,
        '--num_workers',             '2',
        '--joint_spec_name',         'coco18',
        '--use_two_stream',
        '--model_depth',             str(DEPTH),
        '--loss_name',               'focal',
        '--focal_alpha_mode',        'sqrt_inverse',
        '--bbox_norm',
        '--center_norm',
        '--weight_decay',            '0.0005',
        '--save_every_epochs',       '10',
    ]
    import scripts.train_gym99 as _train
    importlib.reload(_train)
    print(f'>>> Training {EID} — Two-Stream bare depth={DEPTH}...')
    _train.main()
    print(f'\n✅ {EID} done.')

In [ ]:
# ── E6c: Two-Stream bare, depth=4 ─────────────────────────────────────────────
import importlib

DEPTH = 4
EID   = E6_IDS[DEPTH]
OUT   = E6_DIRS[DEPTH]
checkpoint = Path(OUT) / f'stgcn_gym99_coco18_2s_d{DEPTH}_expert_FX.pth'
if checkpoint.exists():
    print(f'[{EID}] Checkpoint found, skipping: {checkpoint}')
else:
    sys.argv = [
        'train_gym99.py',
        '--auto_build_from_gym288',
        '--gym288_dataset_path',     GYM288_PKL,
        '--dataset_path',            GYM99_PKL,
        '--out_dir',                 OUT,
        '--apparatus',               'FX',
        '--epochs',                  str(EPOCHS),
        '--batch_size',              str(BATCH_SIZE),
        '--lr',                      LR,
        '--early_stopping_patience', EARLY_STOPPING_PATIENCE,
        '--num_workers',             '2',
        '--joint_spec_name',         'coco18',
        '--use_two_stream',
        '--model_depth',             str(DEPTH),
        '--loss_name',               'focal',
        '--focal_alpha_mode',        'sqrt_inverse',
        '--bbox_norm',
        '--center_norm',
        '--weight_decay',            '0.0005',
        '--save_every_epochs',       '10',
    ]
    import scripts.train_gym99 as _train
    importlib.reload(_train)
    print(f'>>> Training {EID} — Two-Stream bare depth={DEPTH}...')
    _train.main()
    print(f'\n✅ {EID} done.')

---
## Phần 2: So sánh kết quả

Các cell dưới load `history.json` và `metrics_train_gym99.json` của mỗi experiment để vẽ và so sánh.

In [ ]:
# ── Cell: Load tất cả histories ──────────────────────────────────────────────
import json

# E0 Baseline — cần cung cấp path thủ công hoặc điền số liệu tham chiếu
# Nếu có history.json từ Baseline-result.ipynb, hãy cập nhật path dưới đây:
BASELINE_HISTORY_PATH = os.getenv('BASELINE_HISTORY_PATH', '')

HISTORIES = {}  # {exp_id: history_dict}
METRICS   = {}  # {exp_id: metrics_dict}

# Load E0 nếu có
if BASELINE_HISTORY_PATH and Path(BASELINE_HISTORY_PATH).exists():
    with open(BASELINE_HISTORY_PATH) as f:
        HISTORIES['E0'] = json.load(f)
    print(f'[E0] Loaded baseline history from {BASELINE_HISTORY_PATH}')
else:
    # Điền thủ công từ Baseline-result.ipynb nếu không có file
    print('[E0] Baseline history not found — bỏ qua hoặc điền thủ công vào HISTORIES["E0"]')

# Load E1 – E5
for eid in EXP_IDS:
    h_path = Path(EXP_DIRS[eid]) / 'history.json'
    m_path = Path(EXP_DIRS[eid]) / 'metrics_train_gym99.json'
    if h_path.exists():
        with open(h_path) as f:
            HISTORIES[eid] = json.load(f)
        print(f'[{eid}] Loaded history ({len(HISTORIES[eid]["val_acc"])} epochs)')
    else:
        print(f'[{eid}] history.json not found — run training cell first.')
    if m_path.exists():
        with open(m_path) as f:
            METRICS[eid] = json.load(f)

print(f'\nLoaded histories: {list(HISTORIES.keys())}')

In [ ]:
# ── Load histories cho nhóm E6 ────────────────────────────────────────────────
for d in E6_DEPTHS:
    eid = E6_IDS[d]
    h_path = Path(E6_DIRS[d]) / 'history.json'
    m_path = Path(E6_DIRS[d]) / 'metrics_train_gym99.json'
    if h_path.exists():
        with open(h_path) as f:
            HISTORIES[eid] = json.load(f)
        print(f'[{eid}] Loaded history ({len(HISTORIES[eid]["val_acc"])} epochs)')
    else:
        print(f'[{eid}] history.json not found.')
    if m_path.exists():
        with open(m_path) as f:
            METRICS[eid] = json.load(f)

print(f'\nAll loaded experiments: {list(HISTORIES.keys())}')

In [ ]:
# ── Two-Stream bare vs Baseline & E1 — focused comparison ─────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

focus_ids = ['E0', 'E1'] + [E6_IDS[d] for d in E6_DEPTHS if E6_IDS[d] in HISTORIES]
focus_ids = [e for e in focus_ids if e in HISTORIES]

if len(focus_ids) < 2:
    print('Cần ít nhất 2 experiments để so sánh.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle('Two-Stream bare vs Baseline — Does Two-Stream help without tricks?',
                 fontsize=12, fontweight='bold')

    for ax, (key, title) in zip(axes, [('val_acc', 'Val Accuracy'), ('val_loss', 'Val Loss')]):
        for eid in focus_ids:
            h = HISTORIES[eid]
            if key not in h:
                continue
            epochs = range(1, len(h[key]) + 1)
            ax.plot(
                epochs, h[key],
                color=COLORS.get(eid, 'black'),
                linestyle=STYLES.get(eid, '-'),
                linewidth=WIDTHS.get(eid, 1.5),
                label=EXP_LABELS.get(eid, eid),
            )
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('Epoch')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    plt.tight_layout()
    out_png = Path(OUT_BASE) / 'twostream_bare_vs_baseline.png'
    plt.savefig(out_png, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_png}')

    # Bảng nhỏ
    print(f'\n{"Exp":<6}  {"Mô tả":<40}  {"Best Val":>10}  {"Delta vs E1":>12}')
    print('-' * 75)
    base = max(HISTORIES['E1']['val_acc']) if 'E1' in HISTORIES else 0
    for eid in focus_ids:
        best  = max(HISTORIES[eid]['val_acc'])
        delta = best - base
        print(f'{eid:<6}  {EXP_LABELS.get(eid, eid):<40}  {best:>10.4f}  {delta:>+12.4f}')

In [ ]:
# ── Cell: Bảng tổng kết định lượng ───────────────────────────────────────────
import json

EXP_LABELS = {
    'E0': 'E0 — Baseline (depth=10, CE)',
    'E1': 'E1 — Focal Loss (depth=6)',
    'E2': 'E2 — +Warm-up & Clip',
    'E3': 'E3 — +Weighted Sampler',
    'E4': 'E4 — +Augmentation',
    'E5': 'E5 — Two-Stream (full)',
}

print(f'{"Exp":<5}  {"Mô tả":<35}  {"Best Val Acc":>12}  {"@Epoch":>7}  '
      f'{"Final Val":>10}  {"Final Train":>12}  {"Gap":>8}  {"Macro F1":>9}')
print('-' * 100)

for eid in ['E0', 'E1', 'E2', 'E3', 'E4', 'E5', 'E6a', 'E6b', 'E6c']:
    if eid not in HISTORIES:
        print(f'{eid:<5}  {EXP_LABELS.get(eid, eid):<35}  {"(no data)":>12}')
        continue
    h = HISTORIES[eid]
    best_val   = max(h['val_acc'])
    best_epoch = h['val_acc'].index(best_val) + 1
    final_val  = h['val_acc'][-1]
    final_tr   = h['train_acc'][-1]
    gap        = final_tr - final_val

    macro_f1 = '—'
    if eid in METRICS:
        macro_f1 = f"{METRICS[eid].get('macro_f1', 0):.4f}"

    label = EXP_LABELS.get(eid, eid)
    print(f'{eid:<5}  {label:<35}  {best_val:>12.4f}  {best_epoch:>7}  '
          f'{final_val:>10.4f}  {final_tr:>12.4f}  {gap:>+8.4f}  {macro_f1:>9}')

In [ ]:
# ── Cell: Training curves — Val Accuracy & Val Loss cho tất cả experiments ────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

COLORS = {
    'E0': '#95a5a6',   # xám — baseline
    'E1': '#e74c3c',   # đỏ
    'E2': '#e67e22',   # cam
    'E3': '#f1c40f',   # vàng
    'E4': '#27ae60',   # xanh lá
    'E5': '#2980b9',   # xanh dương — full pipeline
}
STYLES = {'E0': ':', 'E1': '--', 'E2': '-.', 'E3': '--', 'E4': '-', 'E5': '-'}
WIDTHS = {'E0': 1.5, 'E1': 1.5, 'E2': 1.5, 'E3': 1.5, 'E4': 2.0, 'E5': 2.5}

if not HISTORIES:
    print('No histories loaded — run training and load cells first.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Ablation Study — FX Subset (35 classes)', fontsize=15, fontweight='bold')

    for ax, (key, title) in zip(axes, [('val_acc', 'Val Accuracy'), ('val_loss', 'Val Loss')]):
        for eid, h in HISTORIES.items():
            if key not in h:
                continue
            epochs = range(1, len(h[key]) + 1)
            label = EXP_LABELS.get(eid, eid)
            ax.plot(
                epochs, h[key],
                color=COLORS.get(eid, 'black'),
                linestyle=STYLES.get(eid, '-'),
                linewidth=WIDTHS.get(eid, 1.5),
                label=label,
            )
        ax.set_title(title, fontsize=12)
        ax.set_xlabel('Epoch')
        ax.legend(fontsize=9, loc='lower right' if 'acc' in key else 'upper right')
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    plt.tight_layout()
    out_png = Path(OUT_BASE) / 'ablation_curves.png'
    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_png, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_png}')

In [ ]:
# ── Cell: Overfitting gap — so sánh Train vs Val accuracy ─────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

if not HISTORIES:
    print('No histories loaded.')
else:
    n = len(HISTORIES)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4.5), sharey=True)
    if n == 1:
        axes = [axes]
    fig.suptitle('Train vs Val Accuracy — Overfitting Gap per Experiment', fontsize=13, fontweight='bold')

    for ax, (eid, h) in zip(axes, HISTORIES.items()):
        epochs = range(1, len(h['train_acc']) + 1)
        ax.plot(epochs, h['train_acc'], color='steelblue', linewidth=1.8, label='Train')
        ax.plot(epochs, h['val_acc'],   color='tomato',    linewidth=1.8, label='Val')
        ax.fill_between(epochs, h['val_acc'], h['train_acc'],
                        alpha=0.15, color='orange', label='Gap')

        best_val   = max(h['val_acc'])
        best_epoch = h['val_acc'].index(best_val) + 1
        ax.axvline(best_epoch, color='gray', linestyle='--', linewidth=1, alpha=0.7)
        label = EXP_LABELS.get(eid, eid)
        ax.set_title(f'{label}\nbest val={best_val:.3f} @ep{best_epoch}', fontsize=9)
        ax.set_xlabel('Epoch')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    axes[0].set_ylabel('Accuracy')
    plt.tight_layout()
    out_png = Path(OUT_BASE) / 'ablation_gap.png'
    plt.savefig(out_png, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_png}')

In [ ]:
# ── Cell: Bar chart — Best Val Accuracy per experiment ───────────────────────
import matplotlib.pyplot as plt
import numpy as np

if not HISTORIES:
    print('No histories loaded.')
else:
    eids   = [eid for eid in ['E0','E1','E2','E3','E4','E5','E6a','E6b','E6c'] if eid in HISTORIES]
    labels = [EXP_LABELS.get(e, e) for e in eids]
    bests  = [max(HISTORIES[e]['val_acc']) for e in eids]
    colors = [COLORS.get(e, 'gray') for e in eids]

    # Shorten labels for bar chart
    short_labels = [e for e in eids]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(short_labels, bests, color=colors, edgecolor='black', linewidth=0.6)
    ax.set_title('Best Val Accuracy — Ablation Study (FX 35 classes)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Best Val Accuracy')
    ax.set_ylim(0, min(1.0, max(bests) * 1.15))
    ax.grid(axis='y', alpha=0.3)

    for bar, val, eid in zip(bars, bests, eids):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    # Add full labels as x-tick tooltips (rotated)
    ax.set_xticks(range(len(eids)))
    ax.set_xticklabels([f'{eid}\n{EXP_LABELS[eid].split("—")[1].strip()[:30]}'
                        for eid in eids], fontsize=8)

    plt.tight_layout()
    out_png = Path(OUT_BASE) / 'ablation_bar.png'
    plt.savefig(out_png, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_png}')

In [ ]:
# ── Cell: Delta improvement — mức tăng so với E1 (Focal Loss baseline) ────────
import matplotlib.pyplot as plt
import numpy as np

if 'E1' not in HISTORIES:
    print('[skip] E1 not loaded — cần E1 làm base để tính delta.')
else:
    base_val = max(HISTORIES['E1']['val_acc'])
    print(f'Reference (E1) best val acc = {base_val:.4f}\n')
    print(f'{"Exp":<5}  {"Mô tả":<35}  {"Best Val":>10}  {"Delta vs E1":>12}')
    print('-' * 68)
    for eid in ['E0', 'E1', 'E2', 'E3', 'E4', 'E5', 'E6a', 'E6b', 'E6c']:
        if eid not in HISTORIES:
            continue
        best = max(HISTORIES[eid]['val_acc'])
        delta = best - base_val
        sign = '+' if delta >= 0 else ''
        label = EXP_LABELS.get(eid, eid)
        marker = ' ◀ reference' if eid == 'E1' else ('  ★ BEST' if best == max(max(HISTORIES[e]['val_acc']) for e in HISTORIES) else '')
        print(f'{eid:<5}  {label:<35}  {best:>10.4f}  {sign}{delta:>+11.4f}{marker}')